# spatial trajectories synthetic dataset benchmark

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Score the frozen endpoint cohort and render Figure 4C–D

In [ ]:
# Zoetrope synthetic crescent: Results Figure, Panel B.
# Top: pooled physical endpoint NRMSE across nine future tissues.
# Bottom: widened Tissue 10 endpoint comparison (zero-based tissue ID 9; 10 spatially spread source cells).
# ONLY inference-time endpoint methods: Zoetrope + arclength, stVCR, moscot.
# No oracle contour or oracle-adjusted predictions are opened or plotted.
# Important: common-success cohort is selected ONLY from finite predictions,
# before hidden truth is opened; failures and coverage are reported separately.
# Frozen synthetic dataset seed 20261023; no model training or tuning.

from pathlib import Path
import os, json, hashlib
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

# ----------------------- Paths and settings -----------------------
PROJECT = Path(os.environ.get(
    'STP_PROJECT', os.environ["STP_PROJECT"]
)).expanduser()
DATA = Path(os.environ.get(
    'CRESCENT_DATA_ROOT', str(PROJECT / 'Experiments/simulation validation/'
                                     'Synthetic benchmarking/experiments/'
                                     'synthetic_developmental_crescent_v1')
)).expanduser()
BASE = Path(os.environ.get(
    'SYNTHETIC_DISPLACEMENT_ROOT',
    str(PROJECT / 'experiments/synthetic_benchmark/'
                  'synthetic_developmental_crescent_v1_displacement_final_v1')
)).expanduser()
OUT = Path(os.environ.get(
    'SYNTHETIC_TRAJECTORY_PANEL_OUT',
    str(BASE / 'publication_results/results_figure_panel_B_arclength_colored_smallX_wide_v3')
)).expanduser()
OUT.mkdir(parents=True, exist_ok=True)

FIG_W, FIG_H = 4.40, 5.50
DPI = 450
STEM = 'FIG_synthetic_results_panel_B_trajectory_arclength_colored_smallX_T10_wide'
TARGET = 9  # Zero-based synthetic tissue ID; 10th tissue = t=1.
N_DISPLAY = 10
INK, GRID = '#23313B', '#D6DCE1'
RED, ORANGE, GREEN = '#D62728', '#F58518', '#54A24B'
COLORS = {'Zoetrope': RED, 'stVCR': ORANGE, 'moscot': GREEN}

# ----------------------- File helpers -----------------------
def need(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing frozen input: {path}')
    return path

def sha256(path):
    h = hashlib.sha256()
    with open(need(path), 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

# ----------------------- Load frozen predictions -----------------------
manifest = json.loads(need(DATA / 'manifests/dataset_manifest.json').read_text())
assert manifest['dataset_id'] == 'synthetic_developmental_crescent_v1'
assert int(manifest['seed']) == 20261023 and manifest['status'] == 'frozen'
public_rel = 'train_only/public/observed_cells.csv.gz'
public_path = need(DATA / public_rel)
assert sha256(public_path) == manifest['file_sha256'][public_rel]
obs = pd.read_csv(public_path, usecols=['cell_id', 'tissue_id', 'x', 'y'])
obs['cell_id'] = obs['cell_id'].astype(str)
assert obs.cell_id.is_unique and obs.tissue_id.nunique() == 10

forecast_path = need(BASE / 'frozen_public_native_and_adapter_endpoints.npz')
with np.load(forecast_path, allow_pickle=False) as z:
    ids = z['source_ids'].astype(str)
    times = z['times'].astype(float)
    forecasts = {
        'Zoetrope': np.asarray(z['flow_xy'], dtype=float),
        'stVCR': np.asarray(z['stvcr_xy'], dtype=float),
        'moscot': np.asarray(z['moscot_xy'], dtype=float),
    }
assert ids.shape == (180,) and len(set(ids)) == len(ids)
np.testing.assert_allclose(times, np.arange(10) / 9., atol=1e-9, rtol=0)
for name, pred in forecasts.items():
    assert pred.shape == (10, len(ids), 2), (name, pred.shape)

# Appendix E.1: freeze eligibility before opening any evaluator-only truth.
# The auxiliary transverse adapter defines coverage, but is not scored here.
valid = {name: np.isfinite(pred).all(axis=2) for name, pred in forecasts.items()}
display_valid = np.logical_and.reduce(list(valid.values()))
auxiliary_path = need(BASE / 'geometry_only_harmonic_transverse_all_tissues_v1' /
                      'geometry_only_transverse_all_tissues_PUBLIC_ONLY.npz')
with np.load(auxiliary_path, allow_pickle=False) as auxiliary:
    np.testing.assert_array_equal(auxiliary['source_ids'].astype(str), ids)
    auxiliary_xy = np.asarray(auxiliary['pred_xy'], dtype=float)
assert auxiliary_xy.shape == (10, len(ids), 2)
common = display_valid & np.isfinite(auxiliary_xy).all(axis=2)
common[0, :] = False
assert common.sum() == 1403, 'Frozen 1,403-pair cohort changed'
assert display_valid[TARGET].sum() >= N_DISPLAY, 'Fewer than ten valid display endpoints'
target_ids, source_rows = np.nonzero(common)
pd.DataFrame({
    'target_tissue_id': target_ids,
    'source_cell_id': ids[source_rows],
}).to_csv(OUT / 'figure_4c_cohort.csv', index=False)

# Deterministic, truth-independent farthest-point sampling from source geometry.
src_public = obs.loc[obs.tissue_id.eq(0)].set_index('cell_id').loc[ids]
source_xy = src_public[['x', 'y']].to_numpy(dtype=float)
pool = np.flatnonzero(display_valid[TARGET])
xy = source_xy[pool]
selected_local = [int(np.argmin(((xy - xy.mean(axis=0))**2).sum(axis=1)))]
while len(selected_local) < N_DISPLAY:
    chosen_xy = xy[np.array(selected_local)]
    distances = ((xy[:, None, :] - chosen_xy[None, :, :])**2).sum(axis=2)
    distance_to_chosen = distances.min(axis=1)
    distance_to_chosen[selected_local] = -np.inf
    selected_local.append(int(np.argmax(distance_to_chosen)))
sel = pool[np.asarray(selected_local, dtype=int)]
chosen_ids = ids[sel]

# ----------------------- Hidden truth for scoring/visualization ONLY -----------------------
endpoint_path = need(BASE / 'per_cell_spatial_endpoints.csv.gz')
per = pd.read_csv(endpoint_path)
per['cell_id'] = per['cell_id'].astype(str)
assert not per.duplicated(['tissue_id', 'cell_id']).any()
truth_rel = 'evaluator_only/observed_hidden/cell_truth.csv.gz'
truth_path = need(DATA / truth_rel)
# Verify hidden truth identity when the manifest records it.
if truth_rel in manifest.get('file_sha256', {}):
    assert sha256(truth_path) == manifest['file_sha256'][truth_rel]
hidden = pd.read_csv(truth_path, usecols=['cell_id','translation_x','translation_y'])
hidden['cell_id'] = hidden['cell_id'].astype(str)
assert hidden.cell_id.is_unique
hix = hidden.set_index('cell_id')

pose = {}
for k in range(10):
    keys = obs.loc[obs.tissue_id.eq(k), 'cell_id']
    offsets = hix.loc[keys, ['translation_x','translation_y']].to_numpy(float)
    assert np.isfinite(offsets).all() and np.ptp(offsets, axis=0).max() < 1e-8
    pose[k] = offsets[0]

src = per.loc[per.tissue_id.eq(0)].set_index('cell_id').loc[ids]
source_phys = src[['source_x', 'source_y']].to_numpy(float)
np.testing.assert_allclose(source_xy - pose[0], source_phys, atol=1e-5, rtol=0)

true_end = {}
squared_displacement = {}
for k in range(1, 10):
    frame = per.loc[per.tissue_id.eq(k)].set_index('cell_id').loc[ids]
    true_end[k] = frame[['true_x','true_y']].to_numpy(float)
    np.testing.assert_allclose(frame[['source_x','source_y']], source_phys, atol=1e-5, rtol=0)
    # Check native frozen endpoint arrays equal the benchmark's endpoint receipt.
    for name, columns in [('Zoetrope',['flow_x','flow_y']),
                          ('stVCR',['stvcr_x','stvcr_y']),
                          ('moscot',['moscot_x','moscot_y'])]:
        np.testing.assert_allclose(forecasts[name][k], frame[columns].to_numpy(float),
                                   atol=1e-5, rtol=0, equal_nan=True)
    squared_displacement[k] = ((true_end[k] - pose[k] - source_phys)**2).sum(axis=1)
    assert np.isfinite(true_end[k]).all() and np.isfinite(squared_displacement[k]).all()

# ----------------------- Paired endpoint accuracy and coverage -----------------------
rows = []
for k in range(1,10):
    matched = common[k]
    denominator = float(squared_displacement[k][matched].sum())
    assert denominator > 0
    for name, pred in forecasts.items():
        numerator = float(((pred[k, matched] - true_end[k][matched])**2).sum())
        rows.append(dict(tissue_id=k, method=name, n_pairs=int(matched.sum()),
                         squared_error=numerator, squared_displacement=denominator,
                         tissue_NRMSE=float(np.sqrt(numerator / denominator))))
by_tissue = pd.DataFrame(rows)
pooled = (by_tissue.groupby('method',sort=False)
    .agg(squared_error=('squared_error','sum'),
         squared_displacement=('squared_displacement','sum'),
         n_pairs=('n_pairs','sum')).reset_index())
pooled['pooled_NRMSE'] = np.sqrt(pooled.squared_error / pooled.squared_displacement)
pooled['common_success_coverage'] = pooled.n_pairs / (9*len(ids))
assert pooled.n_pairs.eq(1403).all(), 'Frozen 1,403-pair cohort changed'
np.testing.assert_allclose(pooled.squared_displacement,
                           pooled.squared_displacement.iloc[0],atol=1e-9,rtol=0)

availability = pd.DataFrame([{
    'method':name,
    'finite_prediction_pairs':int(valid[name][1:].sum()),
    'total_possible_pairs':9*len(ids),
    'method_endpoint_coverage':float(valid[name][1:].mean()),
    'common_success_pairs':int(common[1:].sum()),
    'common_success_coverage':float(common[1:].mean())
} for name in forecasts])

print('\nPAIRED non-oracle-only pooled endpoint NRMSE (lower better):')
print(pooled[['method','pooled_NRMSE','n_pairs','common_success_coverage']].to_string(index=False))
print('\nPer-method endpoint coverage:')
print(availability[['method','finite_prediction_pairs','method_endpoint_coverage']].to_string(index=False))

# ----------------------- Typography -----------------------
installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((f for f in ('Arial','Liberation Sans','Arimo','Nimbus Sans','DejaVu Sans')
             if f in installed), 'DejaVu Sans')
print('Publication font:', FONT)
mpl.rcParams.update({
    'font.family':FONT, 'font.size':10.5, 'axes.labelsize':11.5,
    'xtick.labelsize':10.2,'ytick.labelsize':10.5,
    'axes.linewidth':0.75,'axes.edgecolor':INK,'axes.labelcolor':INK,
    'xtick.color':INK,'ytick.color':INK,'text.color':INK,
    'pdf.fonttype':42,'ps.fonttype':42,'svg.fonttype':'path',
    'savefig.bbox':None, 'figure.facecolor':'white',
    'axes.facecolor':'white','savefig.facecolor':'white',
})

# ----------------------- Compact vertical panel -----------------------
fig = plt.figure(figsize=(FIG_W,FIG_H),facecolor='white')

# Top: paired pooled NRMSE. Values and full metrics are computed, not hard-coded.
fig.text(.50, 5.31/FIG_H, 'Spatial endpoint prediction', ha='center',va='center',
         fontsize=12.0,fontweight='bold',color=INK)
ax_bar = fig.add_axes([.83/FIG_W, 4.00/FIG_H, 2.79/FIG_W, 1.01/FIG_H])
methods = ['Zoetrope','stVCR','moscot']
vals = pooled.set_index('method').loc[methods,'pooled_NRMSE'].to_numpy(float)
bars = ax_bar.bar(np.arange(3),vals,width=.70,color=[COLORS[m] for m in methods],zorder=3)
ax_bar.set_ylim(0,max(vals)*1.22)
ax_bar.set_xlim(-.48,2.48)
ax_bar.set_xticks(np.arange(3))
ax_bar.set_xticklabels(methods,rotation=36,ha='right',rotation_mode='anchor',
                       fontweight='bold',fontsize=10.2)
ax_bar.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax_bar.set_ylabel('Endpoint NRMSE',fontsize=11.5,labelpad=3)
ax_bar.tick_params(axis='x',length=0,pad=2)
ax_bar.tick_params(axis='y',length=3,width=.7,pad=2)
ax_bar.grid(axis='y',color=GRID,alpha=.5,linewidth=.55)
ax_bar.set_axisbelow(True)
ax_bar.spines[['top','right']].set_visible(False)
for rectangle,v in zip(bars,vals):
    ax_bar.annotate(f'{v:.2f}',(rectangle.get_x()+rectangle.get_width()/2,v),
                    xytext=(0,3),textcoords='offset points',ha='center',va='bottom',
                    fontsize=9.5,color=INK)

# Bottom: final-tissue display. Legend below the tissue uses otherwise empty space.
fig.text(.50, 3.37/FIG_H, f'Tissue {TARGET + 1} · endpoint comparison', ha='center',va='center',
         fontsize=11.5,fontweight='bold',color=INK)

# Four marker conventions: black true-endpoint circles and distinct colored Xs.
# The underlying predictions, cohort, metric, and tissue geometry are unchanged.
legend_handles = [
    Line2D([0],[0],marker='o',color='none',linestyle='None',
           markerfacecolor='#172027',markeredgecolor='white',
           markeredgewidth=.75,markersize=6.5,label='True endpoint'),
    Line2D([0],[0],marker='x',color=RED,linestyle='None',
           markersize=8.5,markeredgewidth=2.1,label='Zoetrope'),
    Line2D([0],[0],marker='x',color=ORANGE,linestyle='None',
           markersize=8.5,markeredgewidth=2.1,label='stVCR'),
    Line2D([0],[0],marker='x',color=GREEN,linestyle='None',
           markersize=8.5,markeredgewidth=2.1,label='moscot'),
]

fig.legend(handles=legend_handles,ncol=2,loc='upper center',
           bbox_to_anchor=(.50,.72/FIG_H),bbox_transform=fig.transFigure,
           frameon=False,fontsize=9.6,columnspacing=.95,handletextpad=.38,
           labelspacing=.30,borderpad=0.)

# Size and anchor the REAL tissue extent, not an oversized fixed subplot box.
# This removes the vacant region immediately below the Tissue 10 heading,
# while preserving equal physical scaling and ALL outlying predictions.
bg = obs.loc[obs.tissue_id.eq(TARGET),['x','y']].to_numpy(float)
true = true_end[TARGET][sel]
span = max(float(np.ptp(bg[:,0])),float(np.ptp(bg[:,1])))
pad = .040*span
displayed = np.vstack([bg, true] + [forecasts[name][TARGET,sel] for name in forecasts])
assert np.isfinite(displayed).all()
X0, X1 = displayed[:,0].min()-pad, displayed[:,0].max()+pad
Y0, Y1 = -displayed[:,1].max()-pad, -displayed[:,1].min()+pad
DX, DY = X1-X0, Y1-Y0
assert DX>0 and DY>0

# A figure-width increase only makes the tissue larger when the plotting
# region matches the actual data's aspect ratio. Cap the height to leave
# sufficient room for the legend and prevent overlap with the upper bar plot.
TISSUE_MAX_W = 4.08  # physical inches; 0.16-inch margin each side
TISSUE_MAX_H = 2.21
TISSUE_TOP = 3.15  # inches from bottom, just under Tissue 10 title
scale = min(TISSUE_MAX_W / DX, TISSUE_MAX_H / DY)
TISSUE_W = DX * scale
TISSUE_H = DY * scale
TISSUE_LEFT = (FIG_W - TISSUE_W)/2
TISSUE_BOTTOM = TISSUE_TOP - TISSUE_H
assert TISSUE_BOTTOM > .84, 'Tissue plot would overlap the legend'
print(f'Tissue plot physical area: {TISSUE_W:.2f} x {TISSUE_H:.2f} inches; '
      f'bottom={TISSUE_BOTTOM:.2f} in, top={TISSUE_TOP:.2f} in')
ax_tissue = fig.add_axes([TISSUE_LEFT/FIG_W, TISSUE_BOTTOM/FIG_H,
                          TISSUE_W/FIG_W, TISSUE_H/FIG_H])
ax_tissue.scatter(bg[:,0],-bg[:,1],s=1.2,color='#AEB6BE',alpha=.42,
                  linewidths=0,rasterized=True,zorder=1)

# Smaller, colored Xs for every method. Using an identical marker shape ensures
# that color encodes method identity and geometry is not obscured by bulky fills.
for name in ('Zoetrope', 'stVCR', 'moscot'):
    p = forecasts[name][TARGET, sel]
    assert np.isfinite(p).all()
    color = COLORS[name]

    # Thin connectors go from the true endpoint to the forecast endpoint.
    for estimate, ground in zip(p, true):
        ax_tissue.plot([ground[0], estimate[0]], [-ground[1], -estimate[1]],
                       color=color, alpha=.55, lw=1.10, zorder=3,
                       rasterized=True)

    ax_tissue.scatter(p[:,0], -p[:,1],
                      s=92, marker='x', color=color,
                      linewidths=2.2, zorder=8)

# Smaller true-endpoint circles with a thin white halo.
ax_tissue.scatter(true[:,0], -true[:,1], s=42, marker='o',
                  color='#172027', edgecolors='white',
                  linewidths=.75, zorder=12)
from matplotlib import patheffects as pe
for j, (xx, yy) in enumerate(true, 1):
    number = ax_tissue.annotate(str(j), (xx, -yy), xytext=(4, 4),
        textcoords='offset points', fontsize=8.2,
        fontweight='bold', color='#172027', zorder=15)
    number.set_path_effects([pe.withStroke(linewidth=2.0,
                                            foreground='white')])

# Already computed the view limits from ALL endpoints before placing axes;
# no method's outlying predictions are cropped away.
ax_tissue.set_xlim(X0, X1)
ax_tissue.set_ylim(Y0, Y1)
ax_tissue.set_aspect('equal',adjustable='box')
ax_tissue.set_axis_off()
if (np.ptp(displayed[:,0]) > 1.5*np.ptp(bg[:,0]) or
        np.ptp(displayed[:,1]) > 1.5*np.ptp(bg[:,1])):
    print('NOTICE: Predictions extend substantially beyond tissue; full endpoint extent retained. '
          'Consider a separate magnified tissue inset if points are crowded.')

# ----------------------- Validate and export -----------------------
fig.canvas.draw()
b = fig.get_tightbbox(fig.canvas.get_renderer()).extents
print('Canvas:',FIG_W,'x',FIG_H,'inches; content bounds:',
      [round(float(v),3) for v in b])
assert b[0]>=-.006 and b[1]>=-.006 and b[2]<=FIG_W+.006 and b[3]<=FIG_H+.006, (
    'Figure content exceeds fixed SVG canvas; inspect labels/legend before exporting')
for ext in ('svg','pdf','png'):
    dest = OUT / f'{STEM}.{ext}'
    fig.savefig(dest,dpi=DPI,bbox_inches=None,facecolor='white')
    print('SAVED:',dest)
plt.show()
plt.close(fig)

# Write audit artifacts separately; NEVER overwrite frozen inputs.
pooled.to_csv(OUT / 'pooled_common_success_three_methods.csv',index=False)
by_tissue.to_csv(OUT / 'per_tissue_common_success_three_methods.csv',index=False)
availability.to_csv(OUT / 'endpoint_coverage_three_methods.csv',index=False)
pd.DataFrame({'display_number':np.arange(1,N_DISPLAY+1),
              'source_cell_id':chosen_ids}).to_csv(OUT/'tissue9_display_cell_ids.csv',index=False)
receipt = {
    'dataset_id':manifest['dataset_id'],'dataset_seed':20261023,
    'panel_inches':[FIG_W,FIG_H],
    'tissue_axes_inches':[TISSUE_W,TISSUE_H],
    'tissue_axes_top_inches':TISSUE_TOP,
    'source_cohort_n':len(ids),'target_tissue_id_zero_based':TARGET,'display_tissue_number':TARGET+1,
    'main_methods':methods,'excluded':'oracle contour, oracle-u, learned-u, transverse',
    'prediction_eligibility':'common non-finite mask across THREE named inference-time methods; selected prior to opening truth',
    'main_bar_metric':'pooled RMSE(endpoint XY) / RMS(true translation-corrected displacement), on paired common-success points across tissue 1..9',
    'coverage_warning':'Arclength may abstain; report common and method-specific coverage; do not conflate with full-coverage NRMSE',
    'n_common_success_pairs':int(common[1:].sum()),
    'n_total_possible_pairs':int(9*len(ids)),
    'tissue9_display_selection':'10 spatially spread source IDs, farthest-point sampling in public tissue0 XY; no error-based selection',
    'tissue9_ids':chosen_ids.tolist(),
    'text_font':FONT,
    'endpoint_marker_style':{'Zoetrope':'red x','stVCR':'orange x',
                             'moscot':'green x','predicted_size':92,
                             'predicted_linewidth':2.2,
                             'true_marker_size':42,'connector_linewidth':1.10},
    'input_hashes':{str(p):sha256(p) for p in (public_path, forecast_path, endpoint_path, truth_path)},
}
(OUT / f'{STEM}_receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
print('Done. Output:',OUT)

# Appendix E.1: apply a public-data centroid shift wherever an endpoint is missing.
initial_centroid = obs.loc[obs.tissue_id.eq(0), ['x', 'y']].to_numpy(float).mean(axis=0)
fallback_rows = []
for tissue_id in range(1, 10):
    target_centroid = obs.loc[obs.tissue_id.eq(tissue_id), ['x', 'y']].to_numpy(float).mean(axis=0)
    fallback_xy = source_xy + target_centroid - initial_centroid
    denominator = squared_displacement[tissue_id].sum()
    for method, prediction in forecasts.items():
        missing = ~valid[method][tissue_id]
        endpoints = np.where(missing[:, None], fallback_xy, prediction[tissue_id])
        fallback_rows.append(dict(
            tissue_id=tissue_id, method=method, n_pairs=len(ids),
            n_fallback=int(missing.sum()), squared_displacement=denominator,
            squared_error=np.square(endpoints - true_end[tissue_id]).sum(),
        ))
fallback_by_tissue = pd.DataFrame(fallback_rows)
fallback_pooled = fallback_by_tissue.groupby('method', sort=False).sum(numeric_only=True).drop(columns='tissue_id')
fallback_pooled['pooled_NRMSE'] = np.sqrt(
    fallback_pooled['squared_error'] / fallback_pooled['squared_displacement']
)
assert fallback_pooled['n_pairs'].eq(1620).all()
fallback_by_tissue.to_csv(OUT / 'public_centroid_fallback_by_tissue.csv', index=False)
fallback_pooled.to_csv(OUT / 'public_centroid_fallback_pooled.csv')
print(fallback_pooled[['n_pairs', 'n_fallback', 'pooled_NRMSE']].to_string())